## 05_data_modeling.ipynb

In [2]:
import pandas as pd

master_dataset = pd.read_csv(
    "../data/curated/master_airbnb_dataset.csv"
)

print(master_dataset.shape)

C:\Users\Sagar\AppData\Local\Temp\ipykernel_26724\3266037049.py:3: DtypeWarning: Columns (0: neighborhood_overview, 1: host_since, 2: host_response_time, 3: host_response_rate, 4: host_acceptance_rate, 5: host_thumbnail_url, 6: host_neighbourhood, 7: host_verifications, 8: neighbourhood, 9: instant_bookable, 10: host_profile_url, 11: neighbourhood_group_cleansed, 12: price_quote_checkin_date, 13: price_quote_checkout_date, 14: price_quote_raw, 15: license) have mixed types. Specify dtype option on import or set low_memory=False.
  master_dataset = pd.read_csv(


(131907, 106)


In [3]:
dim_city = (
    master_dataset[["city"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

dim_city["city_id"] = (
    dim_city.index + 1
)

dim_city

,city,city_id
0,London,1
1,New York,2


In [4]:
dim_neighbourhood = (
    master_dataset[
        ["city", "neighbourhood_cleansed"]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

dim_neighbourhood["neighbourhood_id"] = (
    dim_neighbourhood.index + 1
)

print(dim_neighbourhood.shape)

dim_neighbourhood.head()

(257, 3)


,city,neighbourhood_cleansed,neighbourhood_id
0,London,Islington,1
1,London,Kensington and Chelsea,2
2,London,Westminster,3
3,London,Wandsworth,4
4,London,Tower Hamlets,5


In [6]:
dim_neighbourhood = dim_neighbourhood.merge(
    dim_city,
    on="city",
    how="left"
)

dim_neighbourhood = dim_neighbourhood[
    ["neighbourhood_id", "city_id", "city", "neighbourhood_cleansed"]
]

dim_neighbourhood.head()

,neighbourhood_id,city_id,city,neighbourhood_cleansed
0,1,1,London,Islington
1,2,1,London,Kensington and Chelsea
2,3,1,London,Westminster
3,4,1,London,Wandsworth
4,5,1,London,Tower Hamlets


In [7]:
dim_listing = (
    master_dataset[
        [
            "id",
            "room_type",
            "property_type"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

dim_listing = dim_listing.rename(
    columns={
        "id": "listing_id"
    }
)

print(dim_listing.shape)

dim_listing.head()

(131907, 3)


,listing_id,room_type,property_type
0,13913,Private room,Private room in rental unit
1,15400,Entire home/apt,Entire rental unit
2,17402,Entire home/apt,Entire rental unit
3,24328,Entire home/apt,Entire townhouse
4,36274,Entire home/apt,Entire condo


In [8]:
fact_listing = master_dataset.merge(
    dim_city,
    on="city",
    how="left"
)

In [9]:
fact_listing = fact_listing.merge(
    dim_neighbourhood[
        [
            "city_id",
            "neighbourhood_cleansed",
            "neighbourhood_id"
        ]
    ],
    on=[
        "city_id",
        "neighbourhood_cleansed"
    ],
    how="left"
)

In [10]:
fact_listing = fact_listing[
    [
        "id",
        "city_id",
        "neighbourhood_id",

        "price_clean",
        "availability_rate",
        "occupancy_rate",

        "total_reviews",
        "review_frequency",

        "price_per_bedroom",

        "host_tenure_years"
    ]
]

In [11]:
fact_listing = fact_listing.rename(
    columns={
        "id": "listing_id"
    }
)

print(fact_listing.shape)

fact_listing.head()

(131907, 10)


,listing_id,city_id,neighbourhood_id,price_clean,availability_rate,occupancy_rate,total_reviews,review_frequency,price_per_bedroom,host_tenure_years
0,13913,1,1,70.0,0.906849,0.093151,55.0,3.471382,70.0,15.843836
1,15400,1,2,149.0,0.545205,0.454795,97.0,6.142436,149.0,15.791781
2,17402,1,3,411.0,0.219178,0.780822,56.0,3.564702,137.0,15.709589
3,24328,1,4,NaN,0.805479,0.194521,95.0,5.943606,NaN,15.983562
4,36274,1,5,210.0,0.884932,0.115068,15.0,0.979428,NaN,15.315068


In [13]:
dim_city.to_csv("../data/curated/dim_city.csv", index=False)
dim_neighbourhood.to_csv("../data/curated/dim_neighbourhood.csv", index=False)
dim_listing.to_csv("../data/curated/dim_listing.csv", index=False)
fact_listing.to_csv("../data/curated/fact_listing.csv", index=False)

print("Star schema tables saved successfully.")

Star schema tables saved successfully.
